In [1]:
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForSeq2Seq
from torch.utils.data import Dataset
import random
import os
import json
from tqdm import tqdm
from datetime import datetime
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
# !export TORCH_USE_CUDA_DSA=1

2024-11-03 04:07:52.544186: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-11-03 04:07:52.544233: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-11-03 04:07:52.545846: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-11-03 04:07:52.554739: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
def set_seed(seed=42):
    random.seed(seed)
#     np.random.seed(seed)
#     tf.random.set_seed(seed)
#     torch.manual_seed(seed)
#     torch.cuda.manual_seed(seed)
#     torch.cuda.manual_seed_all(seed)
#     torch.backends.cudnn.deterministic = True
#     torch.backends.cudnn.benchmark = False

set_seed(42)


In [3]:

'''
load model, tokenizer
'''
device = 'cuda' if torch.cuda.is_available() else 'cpu'

def load_model():
    model_name = "Qwen/Qwen2.5-0.5B"

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype="auto",
        device_map="auto"
    )
    model = model.to(dtype=torch.float16)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    print('using device:', device)
#     model.enable_input_require_grads()
    for name, param in model.named_parameters():
        print(f"Parameter: {name}, Type: {param.dtype}")
    
    model.to(device)
    return model, tokenizer


In [4]:
'''
data part
'''
data_dict = {"hotpot_train": "./dataset/hotpotQA/hotpot_train_v1.1.json",
               "hotpot_test": "./dataset/hotpotQA/hotpot_dev_distractor_v1.json",
               "squad_train": "./dataset/squad2.0/train-v2.0.json",
               "squad_test": "./dataset/squad2.0/dev-v2.0.json"
               }

MAX_SEQ_LEN = 5000
MAX_QES_LEN = 100
MAX_TRAIN_NUM = 10000
"""
数据格式
{"title":"", context":"", "question":"", "answer":"", answer_idx:[(start, end), (), ()]}
"""
import copy

def data_augmentation(data, corpus):
    print("doing augmentation now")
    
    cnt = 0
    new_data = []
    for example in tqdm(data):
        new_data.append(copy.deepcopy(example))
        context = copy.deepcopy(example["context"])
#         if random.random() < 0.3:
#             new_context = ""
#             for idx_pair in example["answer_idx"]:

#                 new_context += context[idx_pair[0]:idx_pair[1]]
#         else:
        new_context = ""
        supporting = []
        for idx_pair in example["answer_idx"]:
            replace_content = random.sample(corpus, 1)[0]
#                 print(type(replace_content))
            new_context += replace_content + context[idx_pair[0]:idx_pair[1]]
            supporting.append(context[idx_pair[0]:idx_pair[1]])
        
        if len(new_context) > MAX_SEQ_LEN - MAX_QES_LEN:
            continue
            
#         example["old_context"] = example["context"]
        example["context"] = new_context
        if cnt == 0:
            
            print("**************sample example**************")
            print("question:", example["question"])
            print("answer:", example["answer"])
#             print("supporting_fact:", supporting)
#             print("context:", context)
            print("before:", new_data[-1]["context"])
            print("after:", example["context"])
            cnt +=1
        new_data.append(example)
    return new_data

In [5]:
                
def hotpotQA_dataload(file_path, da = False):
    data_list = []
    max_len = 0
    corpus = []
    
    with open(file_path) as f:
        text = json.loads(f.read())
        for idx, item in tqdm(enumerate(text)):

            ans = item["answer"]
            question = item["question"]
            title = [t[0] for t in item["context"]]
            context = ["".join(t[1]) for t in item["context"]]
            corpus.extend(context)
            title2context = {}
            for _t, _c in zip(title, context):
                title2context[_t] = _c
            context = "".join(context)
            if len(context) > MAX_SEQ_LEN - MAX_QES_LEN or len(question) > MAX_QES_LEN:
                continue
            supporting_sentence = ["".join(title2context[t[0]]) for t in item["supporting_facts"]]

            answer_idx = []
            for sent in supporting_sentence:
                answer_start = context.find(sent)
                answer_idx.append((answer_start, answer_start + len(sent)))
            new_instance = {"title": ".".join(title), "context": context, "question": question, "answer": ans,
                            "answer_idx": answer_idx}
            #             if idx % 100 == 0:
            #                 print(new_instance)
            data_list.append(new_instance)
#             max_len = max(max_len, len(context + question))
    if da:
        data = data_augmentation(data_list, corpus)
        
    train_num = min(MAX_TRAIN_NUM*2 if da else MAX_TRAIN_NUM, len(data_list))
    data_list = random.sample(data_list, k=train_num)
    return data_list, corpus


def squad_dataload(file_path, da=False):
    data_list = []
    max_len = 0
    corpus = []
    
    with open(file_path) as f:
        text = json.loads(f.read())
        data = text["data"]
        for example in tqdm(data):
            title = example["title"]
            for each in example["paragraphs"]:
                context = each["context"]
                if len(context) > MAX_SEQ_LEN - MAX_QES_LEN:
                    continue
                #             print(context)
                for _v in each["qas"]:
                    try:
                        question = _v["question"]
                        if len(question) > MAX_QES_LEN:
                            continue
                        #                     print(_v["answers"])
                        answer = _v["answers"][0]["text"]
                        answer_start = _v["answers"][0]["answer_start"]
                        while answer_start >=0 and context[answer_start] not in [".", "?", "!"]:
                            answer_start -= 1
                        answer_start +=1
                        new_instance = {"title": title, "context": context, "question": question, "answer": answer,
                                        "answer_idx": [(answer_start, len(context))]}
                        #                 print(new_instance)
                        data_list.append(new_instance)
                        max_len = max(max_len, len(context + question))
                        corpus.append(context[:answer_start])

                    except Exception as e:
                        pass
    if da:
        data = data_augmentation(data_list, corpus)
        
    train_num = min(MAX_TRAIN_NUM*2 if da else MAX_TRAIN_NUM, len(data_list))
    data_list = random.sample(data_list, k=train_num)
    return data_list, corpus

class CustomDataset(Dataset):
    def __init__(self, name, tokenizer, post="train", da = False):
        self.data, _ = self.dataloader(data_dict[f"{name}_{post}"], da)
        self.tokenizer = tokenizer
        self.input_data = []

        self.end_token_ids = self.tokenizer.convert_tokens_to_ids('<|im_end|>')
        prompt1 = "<|im_start|>Please answer the question according to the given context.\ncontext:"
        self.prompt1_tokens = self.tokenizer(prompt1, add_special_tokens=False)

        prompt2 = "\nquestion:"
        self.prompt2_tokens = self.tokenizer(prompt2, add_special_tokens=False)
        
        prompt3 = "\nanswer:"
        self.prompt3_tokens = self.tokenizer(prompt3, add_special_tokens=False)

        #         print(self.tokenizer.tokenize(prompt2))
        #         print(self.prompt2_tokens)

        self.tokenize()
    

    def dataloader(self, file_path, da = False):
        write_path = file_path+".train_"+str(int(da))
        data, corpus = [], []
        if os.path.exists(write_path):
            print("loading from:{}".format(write_path))
            with open(write_path) as f:
                for line in tqdm(f):
                    data.append(json.loads(line.strip()))
                
        else:
            print("loading from:{}".format(file_path))

            if "squad" in file_path:
                data, corpus = squad_dataload(file_path, da)

            elif "hotpot" in file_path:
                data, corpus = hotpotQA_dataload(file_path, da)
            else:
                print("invalid data data, please select from (hotpotQA, squad2.0)")
                return
                
            with open(write_path, "w") as f:
                for item in data:
                    f.write(json.dumps(item)+"\n")
                print("data feature write in :", write_path)

        #     print(data[:10])
        if "train" in file_path:
            print("train_num:{}".format(len(data)))
        else:
            print("eval_num:{}".format(len(data)))
        return data, corpus
    
    def tokenize(self):
        print("begin tokenize")
        for idx, item in enumerate(tqdm(self.data)):
            context_tokens = self.tokenizer(item["context"], add_special_tokens=False)
            question_tokens = self.tokenizer(item["question"], add_special_tokens=False)
            label_tokens = self.tokenizer(item["answer"], add_special_tokens=False)

            max_context_len = MAX_SEQ_LEN - len(self.prompt1_tokens["input_ids"]) - len(self.prompt2_tokens)
            max_question_len = MAX_QES_LEN - 1 - len(self.prompt3_tokens["input_ids"])

            input_ids = (
                    self.prompt1_tokens["input_ids"] +
                    context_tokens["input_ids"][:max_context_len] +
                    self.prompt2_tokens["input_ids"] +
                    question_tokens["input_ids"][:max_question_len] +
                    self.prompt3_tokens["input_ids"] +
                    label_tokens["input_ids"] +
                    [self.end_token_ids]
            )
#             print("input_ids:", input_ids)
            
            if idx == 0:
                print("example:")
                print(self.tokenizer.decode(input_ids))
            attention_mask = (
                    self.prompt1_tokens["attention_mask"] +
                    context_tokens["attention_mask"][:max_context_len] +
                    self.prompt2_tokens["attention_mask"] +
                    question_tokens["attention_mask"][:max_question_len] +
                    self.prompt3_tokens["attention_mask"] +
                    label_tokens["attention_mask"] +
                    [1]
            )

            labels = (
                    [-100] * (len(input_ids) - len(label_tokens["input_ids"]) - 1)
                    + label_tokens["input_ids"]
                    + [self.end_token_ids]
            )
            
            self.input_data.append({"input_ids": input_ids,
                                    "attention_mask": attention_mask,
                                    "labels": labels})
        print("tokenization done!")

    def __getitem__(self, idx):
        # 返回一个字典，包含 input_ids, attention_mask, 以及 labels (如果有的话)
        return self.input_data[idx]

    def __len__(self):
        return len(self.input_data)

In [6]:
def padding_function(batch_tensor, padding_value=None):
    if padding_value == None:
        padding_value = tokenizer.pad_token_id  # 通常使用 tokenizer.pad_token_id

    # 计算最长序列的长度
    max_len = max(len(seq) for seq in batch_tensor)

    # 对每个序列进行左填充
    left_padded_batch = [torch.cat([torch.full((max_len - len(seq),), padding_value), seq]) for seq in batch_tensor]

    # 将左填充后的序列堆叠成一个批量张量
    left_padded_batch = torch.stack(left_padded_batch)
    return left_padded_batch

In [8]:
'''
main
'''
model=None
model, tokenizer = load_model()
all_bfloat16 = all(param.dtype == torch.bfloat16 for param in model.parameters())
print("All parameters are BFloat16:", all_bfloat16)

dataset_name = "squad"
post="train"
da = 0

train_dataset = CustomDataset(dataset_name, tokenizer, post, da)


output_dir = os.path.join("./output/Qwen2_new", dataset_name + "_"+ str(da))
os.makedirs(output_dir, exist_ok=True)
                          
epochs=1
batch_size = 4
gradient_accumulation = 1
lr = 1e-4



total_steps = int(len(train_dataset) * epochs/ batch_size / gradient_accumulation)
print('total steps = {}'.format(total_steps))

optimizer = transformers.AdamW(model.parameters(), lr=lr, correct_bias=True)
# scheduler = transformers.WarmupLinearSchedule(optimizer, warmup_steps = int(total_steps*0.01),
#                                                       t_total=total_steps)
scheduler = transformers.get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps*0.01),  # 可以设置为你需要的warmup步数
    num_training_steps=total_steps
)

print('starting training')
overall_step = 0
running_loss = 0
save_step = 100
                          
for epoch in range(epochs):
    print('epoch {}'.format(epoch + 1))
    now = datetime.now()
#     print('time: {}'.format(now))
    input_data = train_dataset.input_data
    random.shuffle(input_data)
    
    for idx in range(0, len(input_data), batch_size):
        # 获取当前 batch 的数据
        batch_data = input_data[idx: idx + batch_size]

        # 将 batch 数据中的 input_ids 转换为张量，并移动到 GPU
        input_ids_batch = [torch.Tensor(inputs["input_ids"]) for inputs in batch_data]
        input_ids_batch = padding_function(input_ids_batch).to("cuda")
        attention_mask_batch = [torch.Tensor(inputs["attention_mask"]) for inputs in batch_data]
        attention_mask_batch = padding_function(attention_mask_batch, 0).to("cuda")
        labels_batch = [torch.Tensor(inputs["labels"]) for inputs in batch_data]
        labels_batch = padding_function(labels_batch, -1).to("cuda")
        assert torch.max(input_ids_batch) < len(tokenizer.get_vocab())
        assert torch.max(labels_batch) < len(tokenizer.get_vocab())
        assert torch.max(attention_mask_batch) < len(tokenizer.get_vocab())
            
        input_ids_batch = torch.tensor(input_ids_batch).long().to(device)
        attention_mask_batch = torch.tensor(attention_mask_batch).long().to(device)
        labels_batch = torch.tensor(labels_batch).long().to(device)

        #  forward pass
        outputs = model.forward(input_ids=input_ids_batch, attention_mask=attention_mask_batch, labels=labels_batch)
        loss, logits = outputs[:2]

        if gradient_accumulation > 1:
            loss = loss / gradient_accumulation

        #  loss backward
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        
        for name, param in model.named_parameters():
#             if "lora" in name:  # LoRA 层的参数通常带有 "lora" 标记, -> has no gradient
            if param.grad is  None:
                print(f" Layer: {name} has no gradient.")
        #  optimizer step
        if (overall_step + 1) % gradient_accumulation == 0:
            running_loss += loss.item()
            optimizer.step()
            optimizer.zero_grad()
            scheduler.step()
        
                
        if (overall_step + 1) % log_step == 0:
            print('now time: {}:{}. Step {} of piece {} of epoch {}, loss {}'.format(
                datetime.now().hour,
                datetime.now().minute,
                step + 1,
                piece_num,
                epoch + 1,
                running_loss * gradient_accumulation / (log_step / gradient_accumulation)))
            running_loss = 0
        overall_step += 1
        if overall_step % save_step == 0:
            print('saving model for step {}'.format(overall_step))
            if not os.path.exists(output_dir + 'model_step{}'.format(overall_step)):
                os.mkdir(output_dir + 'model_step{}'.format(overall_step))
            model_to_save = model.module if hasattr(model, 'module') else model
            model_to_save.save_pretrained(output_dir + 'model_step{}'.format(overall_step))
            
            print('model_step {} finished'.format(overall_step))

            then = datetime.now()
            print('time: {}'.format(then))
#             print('time for one epoch: {}'.format(then - now))

#     print('training finished')
#     if not os.path.exists(output_dir + 'final_model'):
#         os.mkdir(output_dir + 'final_model')
#     model_to_save = model.module if hasattr(model, 'module') else model
#     model_to_save.save_pretrained(output_dir + 'final_model')

using device: cuda
Parameter: model.embed_tokens.weight, Type: torch.float16
Parameter: model.layers.0.self_attn.q_proj.weight, Type: torch.float16
Parameter: model.layers.0.self_attn.q_proj.bias, Type: torch.float16
Parameter: model.layers.0.self_attn.k_proj.weight, Type: torch.float16
Parameter: model.layers.0.self_attn.k_proj.bias, Type: torch.float16
Parameter: model.layers.0.self_attn.v_proj.weight, Type: torch.float16
Parameter: model.layers.0.self_attn.v_proj.bias, Type: torch.float16
Parameter: model.layers.0.self_attn.o_proj.weight, Type: torch.float16
Parameter: model.layers.0.mlp.gate_proj.weight, Type: torch.float16
Parameter: model.layers.0.mlp.up_proj.weight, Type: torch.float16
Parameter: model.layers.0.mlp.down_proj.weight, Type: torch.float16
Parameter: model.layers.0.input_layernorm.weight, Type: torch.float16
Parameter: model.layers.0.post_attention_layernorm.weight, Type: torch.float16
Parameter: model.layers.1.self_attn.q_proj.weight, Type: torch.float16
Parameter:

RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# vocab = tokenizer.get_vocab()
# print(len(vocab))

In [ ]:
# from bert_score import score

# references = ["this is a test"]
# candidates = ["this is a good test"]
# P, R, F1 = score(candidates, references, lang="en", verbose=True)
# print(P, R, F1)